In [0]:
-- 1. Aseguramos que estamos trabajando en el esquema correcto
USE CATALOG ferreteria_dev;
USE SCHEMA gold;

-- 2. Creamos la mega tabla cruzando todo
CREATE OR REPLACE TABLE gold_ventas_maestra AS
SELECT 
    -- A. Trazabilidad (IDs)
    o.id_orden,
    d.id_detalle,
    
    -- B. Dimensiones de Tiempo (Listas para los filtros del dashboard)
    o.fecha_hora AS fecha_venta,
    YEAR(o.fecha_hora) AS anio_venta,
    MONTH(o.fecha_hora) AS mes_venta,
    WEEKOFYEAR(o.fecha_hora) AS semana_venta,
    DAY(o.fecha_hora) AS dia_venta,
    
    -- C. Dimensiones del Negocio (Quién compra y quién vende)
    c.nombre AS nombre_cliente,
    e.nombre AS nombre_empleado,
    
    -- D. Dimensiones del Producto (Qué se vendió y de qué departamento)
    p.id_producto,
    p.descripcion AS nombre_producto,
    sub.nombre AS nombre_subcategoria,
    cat.nombre AS nombre_categoria,
    
    -- E. Hechos / Métricas (Los números que le importan a tu prima)
    d.cantidad,
    d.precio_unitario_aplicado AS precio_unitario,
    (d.cantidad * d.precio_unitario_aplicado) AS monto_total_linea

FROM ferreteria_dev.silver.ordenes_venta o
-- Unimos el detalle de la venta (el corazón de la transacción)
JOIN ferreteria_dev.silver.ordenes_venta_detalle d 
    ON o.id_orden = d.id_orden
-- Unimos productos para saber qué es
LEFT JOIN ferreteria_dev.silver.productos p 
    ON d.id_producto = p.id_producto
-- Unimos subcategorías y categorías para agrupar
LEFT JOIN ferreteria_dev.silver.subcategorias sub 
    ON p.id_subcategoria = sub.id_subcategoria
LEFT JOIN ferreteria_dev.silver.categorias cat 
    ON sub.id_categoria = cat.id_categoria
-- Unimos cliente y empleado
LEFT JOIN ferreteria_dev.silver.clientes c 
    ON o.id_cliente = c.id_cliente
LEFT JOIN ferreteria_dev.silver.empleados e 
    ON o.id_empleado = e.id_empleado;

In [0]:
-- 1. Aseguramos que estamos trabajando en el esquema correcto
USE CATALOG ferreteria_dev;
USE SCHEMA gold;

-- 2. Creamos la tabla maestra de inventario
CREATE OR REPLACE TABLE gold_movimientos_stock AS
SELECT 
    -- A. Trazabilidad
    m.id_movimiento,
    
    -- B. Dimensiones de Tiempo (Para ver qué días hay más flujo)
    m.fecha_hora AS fecha_movimiento,
    YEAR(m.fecha_hora) AS anio_movimiento,
    MONTH(m.fecha_hora) AS mes_movimiento,
    DAY(m.fecha_hora) AS dia_movimiento,
    
    -- C. El Tipo de Movimiento (Crucial para sumar o restar en el Dashboard)
    m.tipo, -- Suele ser 'entrada', 'salida', 'ajuste', 'merma'
    
    -- D. Dimensiones del Producto (Qué se está moviendo)
    p.id_producto,
    p.descripcion AS nombre_producto,
    sub.nombre AS nombre_subcategoria,
    cat.nombre AS nombre_categoria,
    
    -- E. Hechos / Métricas
    m.cantidad
    
FROM ferreteria_dev.silver.movimientos_inventario m
-- Pegamos el producto para saber qué es
LEFT JOIN ferreteria_dev.silver.productos p 
    ON m.id_producto = p.id_producto
-- Pegamos la jerarquía corregida para poder filtrar por departamento
LEFT JOIN ferreteria_dev.silver.subcategorias sub 
    ON p.id_subcategoria = sub.id_subcategoria
LEFT JOIN ferreteria_dev.silver.categorias cat 
    ON sub.id_categoria = cat.id_categoria;

In [0]:
-- 1. Aseguramos el esquema correcto
USE CATALOG ferreteria_dev;
USE SCHEMA gold;

-- 2. Creamos la tabla maestra de rentabilidad por proveedor
CREATE OR REPLACE TABLE gold_compras_proveedores AS
SELECT 
    -- A. Dimensión del Proveedor
    prov.id_proveedor,
    prov.razon_social AS nombre_proveedor,
    prov.dias_credito,
    
    -- B. Dimensiones del Producto
    p.id_producto,
    p.sku,
    p.descripcion AS nombre_producto,
    p.marca,
    
    -- C. Jerarquía de Departamentos
    sub.nombre AS nombre_subcategoria,
    cat.nombre AS nombre_categoria,
    
    -- D. Métricas de Rentabilidad (Desde la raíz)
    p.precio_compra,
    p.precio_venta,
    (p.precio_venta - p.precio_compra) AS margen_ganancia_pesos,
    ROUND(((p.precio_venta - p.precio_compra) / NULLIF(p.precio_compra, 0)) * 100, 2) AS margen_porcentaje
    
FROM ferreteria_dev.silver.productos p
-- Usamos INNER JOIN para traer solo los productos que ya tienen proveedor asignado
INNER JOIN ferreteria_dev.silver.proveedores prov 
    ON p.id_proveedor = prov.id_proveedor
-- Pegamos la jerarquía
LEFT JOIN ferreteria_dev.silver.subcategorias sub 
    ON p.id_subcategoria = sub.id_subcategoria
LEFT JOIN ferreteria_dev.silver.categorias cat 
    ON sub.id_categoria = cat.id_categoria;